Indice de Adecuación de Capital (CNBS)


In [5]:
import requests
import pandas as pd
from Clave import APIKEY
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


#indicadoresIDs=[338,341,344,347,350,353,356,359,362,365,368,371,374]
indicadoresIDs=[337]
FechaInicio="2024-12-01T00:00:00"
#FechaFinal="2019-12-01T00:00:00"
print(FechaInicio)
df_combinado=[]

for indicadorID in indicadoresIDs:
    urlCifras= f"https://bchapi-am.azure-api.net/api/v1/indicadores/{indicadorID}/cifras"
    urlIndicadores= f"https://bchapi-am.azure-api.net/api/v1/indicadores/{indicadorID}"
    
    params1={
        "fechainicio":FechaInicio
        #,"fechafinal":FechaFinal
    }
    
    headers = {
        "clave" : APIKEY,
        "Conten-Type": "application/json",
    }
    
    responseCifras = requests.get(urlCifras, headers=headers, params=params1, verify=False)
    responseIndicadores = requests.get(urlIndicadores, headers=headers,verify=False)
            
    if responseIndicadores.status_code == 200:
        datosIndicadores = responseIndicadores.json()
        #print(json.dumps(datosIndicadores,indent=4))  
        df_Indicadores=pd.DataFrame([datosIndicadores])
        ##print(df_Indicadores.to_string(index=False))
    else:
        print(f"Error {responseIndicadores.status_code}: {responseIndicadores.text}")
    
    if responseCifras.status_code == 200:
        datos1 = responseCifras.json()
        if datos1:
            df_Cifras=pd.DataFrame(datos1)
        else:
            df_Cifras=pd.DataFrame([{
            "Fecha":FechaInicio,
            "Id":0,       
            "IndicadorId":indicadorID,
            "Nombre":df_Indicadores.loc[0,"Nombre"],
            "Descripcion":df_Indicadores.loc[0,"Descripcion"],
            "Valor":0
            }])            
            ##print(df_Cifras.to_string(index=False))
            # Hacer negativo el valor si Indicador_ID es 113 o 155        
        #df_Cifras.loc[
        #    df_Cifras['IndicadorId'].isin([113, 155]), 'Valor'] = -df_Cifras.loc[df_Cifras['IndicadorId'].isin([113, 155]), 'Valor'
        #].abs()
    else:
        print(f"Error {responseCifras.status_code}: {responseCifras.text}")
  

    df_merge=pd.merge(
        df_Cifras,
        df_Indicadores,
        left_on='IndicadorId',
        right_on='Id',
        how='left'
    )

    #print(df_merge)
    #df_merge['Nombre_Indicador']= 'Indice de Precios al Consumidor IPC'  
    df_merge['NOMBRE_INDICADOR']=df_merge['Descripcion_x'].str.split('-').str[-2]
    df_merge['TIPO']=df_merge['Descripcion_x'].apply(lambda x:'-'.join(x.split('-')[3:5]))
    print(indicadorID, end=",")
    df_final=df_merge[['Fecha','IndicadorId','Periodicidad','Descripcion_x','Valor','NOMBRE_INDICADOR','TIPO']]
    df_combinado.append(df_final)
else:
    print(f"Error {responseCifras.status_code}: {responseCifras.text}") 

df_resultado=pd.concat(df_combinado, ignore_index=True)
df_resultado=df_resultado.rename(columns={
    'Fecha':'FECHA',
    'IndicadorId':'INDICADOR_ID',
    'Periodicidad':'PERIODICIDAD',
    'Descripcion_x':'DESCRIPCION',
    'Valor':'VALOR'
})
##df_merge.to_csv('indicador.csv', index=False, encoding='utf-8-sig')

2024-12-01T00:00:00
337,Error 200: [{"Id":2595173,"IndicadorId":337,"Nombre":"ESR-IMAE-01-4","Descripcion":"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicultura y Pesca - Acumulada","Fecha":"2025-04-01T00:00:00","Valor":2.7804},{"Id":2416245,"IndicadorId":337,"Nombre":"ESR-IMAE-01-4","Descripcion":"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicultura y Pesca - Acumulada","Fecha":"2025-03-01T00:00:00","Valor":3.6772},{"Id":2220420,"IndicadorId":337,"Nombre":"ESR-IMAE-01-4","Descripcion":"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicultura y Pesca - Acumulada","Fecha":"2025-02-01T00:00:00","Valor":2.5185},{"Id":2057245,"IndicadorId":337,"Nombre":"ESR-IMAE-01-4","Descripcion":"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicultura y Pesca - Acumulada","Fecha":"2025-01-01T00:00:00","Valor":2.8517},{"Id":1935129,"IndicadorId":337,"Nombre":"ESR-IMAE-01-4","Descripcion":"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicultura y Pesca - Acumulada","Fecha":"2024-12-01T00:00:00","Valor":-1.1636}]


In [6]:
df_resultado.head(10)

,FECHA,INDICADOR_ID,PERIODICIDAD,DESCRIPCION,VALOR,NOMBRE_INDICADOR,TIPO
0,2025-04-01T00:00:00,337,Mensual,"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicul...",2.7804,"Agricultura, Ganaderia, Silvicultura y Pesca","Agricultura, Ganaderia, Silvicultura y Pesca ..."
1,2025-03-01T00:00:00,337,Mensual,"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicul...",3.6772,"Agricultura, Ganaderia, Silvicultura y Pesca","Agricultura, Ganaderia, Silvicultura y Pesca ..."
2,2025-02-01T00:00:00,337,Mensual,"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicul...",2.5185,"Agricultura, Ganaderia, Silvicultura y Pesca","Agricultura, Ganaderia, Silvicultura y Pesca ..."
3,2025-01-01T00:00:00,337,Mensual,"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicul...",2.8517,"Agricultura, Ganaderia, Silvicultura y Pesca","Agricultura, Ganaderia, Silvicultura y Pesca ..."
4,2024-12-01T00:00:00,337,Mensual,"ESR-IMAE-01 - Agricultura, Ganaderia, Silvicul...",-1.1636,"Agricultura, Ganaderia, Silvicultura y Pesca","Agricultura, Ganaderia, Silvicultura y Pesca ..."


In [ ]:
#df_resultado.drop_duplicates().shape
#df_resultado['TIPO'] = df_resultado['TIPO'].str.strip()
df_resultado['TIPO'] = 'Reservas Monetarias Internacionales'
df_resultado['NOMBRE_INDICADOR'] = df_resultado['NOMBRE_INDICADOR'].str.strip()
df_resultado=df_resultado.drop_duplicates()
df_resultado.head(10)

In [8]:
################           INSERT TABLE / DATA           #################   #      
from azure.identity import InteractiveBrowserCredential
import pandas as pd
from Server import AZURE
from tqdm import tqdm
from sqlalchemy import create_engine, text

credential = InteractiveBrowserCredential()

server = AZURE
database = 'sqlpooldwhandr01'
schema = 'HN_NAP_HO_MISRIESGOS_F'
tabla = 'INDICADORES_MACROECONOMICOS'
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
        f"DRIVER={driver};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
        f"Authentication=ActiveDirectoryInteractive;"
    )

connection_uri = f"mssql+pyodbc:///?odbc_connect={connection_string}"
engine = create_engine(connection_uri, fast_executemany=True)


query = text("""
SELECT COLUMN_NAME, DATA_TYPE , CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = :tablita
AND TABLE_SCHEMA = :esquema
""")

with engine.connect() as conn:
    result = conn.execute(query, {"tablita": tabla, "esquema":schema})
    columns_types = {row[0]: [row[1] , row[2]] for row in result}
#columns_types

OperationalError: (pyodbc.OperationalError) ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]Named Pipes Provider: Could not open a connection to SQL Server [1].  (1) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to SQL Server. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (1)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
#-----------------------------############### INSERT ##################------------------------------------#
#tmp_df = df_cleaned.iloc[:100].copy()


#data_frame = pd.DataFrame(js)
df_resultado['FECHA']=pd.to_datetime(df_resultado['FECHA']).dt.strftime('%Y-%m-%d')
data_frame=df_resultado
chunksize = 100

for start in tqdm(range(0, len(data_frame), chunksize), desc="Insertando datos"):
    end = min(start + chunksize, len(data_frame))
    chunk = data_frame.iloc[start:end]

    try:
        chunk.to_sql(tabla, 
                     con=engine, 
                     schema=schema, 
                     if_exists='append', 
                     index=False, 
                     chunksize=chunksize)
    except Exception as e:
        print(f"Error al insertar datos en la base de datos: {e}")
    #break
print('Proceso de insercion completado')